# Notebook 1: Data Exploration & Knowledge Base ConstructionThis notebook explores the medical insurance dispensing rules dataset,analyzes policy distributions across providers, and structures the raw data into a queryable knowledge base.---

## 1. Load Environment & DatasetLoad dataset directly using relative file paths.

In [1]:
import osimport openpyxlimport pandas as pdimport json# Resolve file path relative to notebooks directoryEXCEL_PATH = os.path.join('..', 'نظم صرف شركات التأمين-2.xlsx')if not os.path.exists(EXCEL_PATH):    EXCEL_PATH = 'نظم صرف شركات التأمين-2.xlsx'wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)ws = wb['Sheet1']print(f'Sheet loaded: {ws.max_row} rows x {ws.max_column} columns')headers = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]print(f'Columns: {headers}')

Sheet loaded: 777 rows x 5 columns
Columns: ['كود العميل', 'العميل', 'البند', 'التفاصيل', 'ملاحظات']


## 2. Structured Data Parsing

In [1]:
data = []for r in range(2, ws.max_row + 1):    row = {        'company_code': str(ws.cell(r, 1).value or '').strip(),        'company_name': str(ws.cell(r, 2).value or '').strip(),        'category': str(ws.cell(r, 3).value or '').strip(),        'details': str(ws.cell(r, 4).value or '').strip(),        'notes': str(ws.cell(r, 5).value or '').strip(),    }    data.append(row)df = pd.DataFrame(data)print(f'Total policy entries: {len(df)}')print(f'Unique insurance entities: {df["company_name"].nunique()}')print(f'Unique rule categories: {df["category"].nunique()}')df.head(5)

Total policy entries: 776
Unique insurance entities: 78
Unique rule categories: 15


## 3. Policy Distribution Analysis

In [1]:
company_counts = df.groupby('company_name')['category'].nunique().reset_index()company_counts.columns = ['Insurance Entity', 'Categories Count']company_counts = company_counts.sort_values('Categories Count', ascending=False)print('=== Top 10 Entities by Policy Categories Coverage ===')for idx, row in company_counts.head(10).iterrows():    print(f"  {row['Insurance Entity']:<45} {row['Categories Count']} categories")

=== Top 10 Entities by Policy Categories Coverage ===
  AXA- EGYPT-اكسا                               14 categories
  EGYCARE-ايجيكير                               14 categories
  GLOBEMED-جلوبميد                              14 categories
  الاهلى للمشروعات الطبية- EL-AHLI              14 categories
  sehaone-صحة وان                               14 categories
  NEXT CARE-نكست كير                            14 categories
  MED RIGHT-ميدرايت                             13 categories
  OMEGA CARE-اوميجا                             13 categories
  اكوهيلث للرعاية الصحية-Eco Health             13 categories
  UNICARE-يونيكير                               13 categories


## 4. Knowledge Base Export

In [1]:
CATEGORY_TRANSLATIONS = {    'نماذج الصرف': 'Dispensing Forms',    'المحظورات': 'Excluded Items',    'التحمل': 'Co-payment',    'التشخيص': 'Diagnosis Requirements',    'صلاحية النموذج': 'Form Validity',    'صورة البطاقة': 'National ID Copy',    'صورة الكارنية': 'Card Copy',    'الختم / إمضاء العميل': 'Stamp & Signature',    'أقصى مدة للصرف': 'Max Dispensing Duration',    'الحد الأقصى': 'Maximum Limit',    'التواصل للموافقات': 'Approval Contacts',    'لينك الاونلاين سيستم': 'Online Portal',    'البدائل': 'Generic Alternatives',    'ملاحظات': 'Notes',}knowledge_base = {}for _, row in df.iterrows():    company = row['company_name']    if company not in knowledge_base:        knowledge_base[company] = {'company_name': company, 'policies': {}}    cat = row['category']    knowledge_base[company]['policies'][cat] = {        'category_ar': cat,        'category_en': CATEGORY_TRANSLATIONS.get(cat, cat),        'details': row['details'],        'notes': row['notes'],    }KB_OUTPUT_PATH = os.path.join('..', 'data', 'insurance_knowledge_base.json')if not os.path.exists(os.path.dirname(KB_OUTPUT_PATH)):    KB_OUTPUT_PATH = 'insurance_knowledge_base.json'with open(KB_OUTPUT_PATH, 'w', encoding='utf-8') as f:    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)print(f'Knowledge base written to: {KB_OUTPUT_PATH}')print(f'Total entities indexed: {len(knowledge_base)}')

Knowledge base written to: ..\data\insurance_knowledge_base.json
Total entities indexed: 78
